En esta etapa de gold tomamos la tabla de transacciones ya limpia y libre de duplicados y generamos tablas factuales y dimensionales

- Clientes (Dim)
- Cotizaciones (Dim)
- Instrumentos (Dim)

- fact_transaction: Transacciones enriquecidas sin agrupaciones
- fact_transaction_daily: Transacciones con granularidad por dia por cada cliente

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_transactions LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.dim_clientes AS (
    SELECT 
        MD5(CONCAT_WS('||', id_cliente)) AS sk_cliente
        ,id_cliente
        ,MIN(fecha) AS fecha_primera_transaccion
        ,MAX(fecha) AS fecha_ultima_transaccion
        ,MIN_BY(sk_transaccion, fecha) AS sk_primera_transaccion
        ,MAX_BY(sk_transaccion, fecha) AS sk_ultima_transaccion
        ,SUM(CASE WHEN tipoTran = 'Compra' THEN 1 ELSE 0 END) AS total_compras
        ,SUM(CASE WHEN tipoTran = 'Venta' THEN 1 ELSE 0 END)  AS total_ventas
        ,COUNT(*) AS total_transacciones
        FROM 
            iol_challenge.silver.deduped_transactions
            GROUP BY id_cliente
)

In [0]:
%sql
SELECT * FROM iol_challenge.gold.dim_clientes LIMIT 10;

Generamos tabla gold con las cotizaciones por simbolo.

Scope: Tomamos cómo criterio listas fijas de cada simbolo para la clasificación. La solución ideal sería usar una API externa para completar la clasificación completa de los instrumentos, lo cuál permitiria análizar de manera completa todo el conjunto de datos.

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.dim_cotizaciones AS
    (
        SELECT 
            MD5(CONCAT_WS('||', simbolo_titulo)) AS sk_simbolo
            ,simbolo_titulo
            ,CASE 
                WHEN
                    simbolo_titulo IN (
                        -- Ley Argentina (Bonares / Ley Local)
                        "AL29", "AL30", "AL35", "AE38", "AL41", "AN29",
                        "AL29D", "AL30D", "AL35D", "AE38D", "AL41D", "AN29D",
                        "AL29C", "AL30C", "AL35C", "AE38C", "AL41C", "AN29C",

                        -- Ley Nueva York (Globales / Ley Extranjera)
                        "GD29", "GD30", "GD35", "GD38", "GD41", "GD46",
                        "GD29D", "GD30D", "GD35D", "GD38D", "GD41D", "GD46D",
                        "GD29C", "GD30C", "GD35C", "GD38C", "GD41C", "GD46C"
                    )
                    THEN "Bono soberano"
                WHEN
                    simbolo_titulo IN (
                        "AAL", "AAPL", "ABBV", "ABT", "ACN", "ADB", "ADE", "ADI", "ADP", "AEM", 
                        "AIG", "AMAT", "AMD", "AMX", "AMZN", "ARCO", "ARKG", "ASML", "AUY", "AVGO", 
                        "BA", "BAC", "BABA", "BBD", "BBV", "BCS", "BIIB", "BITF", "BMA", "BMY", 
                        "BRKB", "C", "CAT", "CBD", "CDE", "CED", "CL", "COIN", "COST", "CRM", 
                        "CSCO", "CVX", "CX", "DE", "DEO", "DIA", "DIS", "EBAY", "EEM", "EFX", 
                        "ERJ", "ETSY", "EWZ", "F", "FCX", "FDX", "FSLR", "GE", "GFI", "GGB", 
                        "GILD", "GLOB", "GM", "GOOGL", "GS", "HAL", "HMC", "HMY", "HOM", "HON", 
                        "HPQ", "HSBC", "IBM", "IFF", "INFY", "INTC", "IP", "IWM", "JNJ", "JPM", 
                        "KMB", "KO", "LFC", "LLY", "LMT", "LRCX", "MA", "MCD", "MELI", "META", 
                        "MMM", "MO", "MRK", "MSFT", "MU", "NEM", "NFLX", "NKE", "NTES", "NVDA", 
                        "ORCL", "PBR", "PEP", "PFE", "PG", "PYPL", "QCOM", "QQQ", "RIO", "ROST", 
                        "SBUX", "SCCO", "SHEL", "SHOP", "SLB", "SNAP", "SNOW", "SPOT", "SPY", "SQ", 
                        "STNE", "T", "TARGET", "TEFO", "TGT", "TM", "TSLA", "TSM", "TX", "TXN", 
                        "UNH", "UNP", "UPST", "V", "VALE", "VIST", "VOD", "VZ", "WBA", "WFC", 
                        "WMT", "X", "XOM", "XP", "YMM", "ZM"
                    ) THEN 'Cedear'
                WHEN 
                    simbolo_titulo IN (
                        "AGRO", "ALUA", "AUSO", "BBAR", "BHIP", "BMA", "BPAT", "BRIO", "BYMA", 
                        "CAPX", "CARC", "CECO2", "CELU", "CEPU", "CGPA2", "COME", "CRES", "CTIO", 
                        "CVH", "DGCU2", "DOME", "DYCA", "EDN", "FERR", "FIPL", "GALA", "GARO", 
                        "GBAN", "GCDI", "GGAL", "GRIM", "HARG", "INTR", "INVJ", "IRS2", "LOMA", 
                        "METR", "MILI", "MOLI", "MORI", "MTR", "OPAR", "PAMP", "PATA", "PGPRI", 
                        "SOMI", "SUPV", "TECO2", "TGNO4", "TGSU2", "TRAN", "TXAR", "VALO", "YPFD"
                    ) THEN 'Accion local'
                ELSE
                    'A revisar'
            END
            AS simbolo_tipo
            ,MIN_BY(sk_transaccion, fecha) AS primera_transaccion_sk
            ,MAX_BY(sk_transaccion, fecha) AS ultima_transaccion_sk
            ,min(FECHA) AS primera_transaccion_fecha
            ,max(fecha) AS ultima_transaccion_fecha
            ,MIN_BY(c.sk_cliente, fecha) AS primera_transaccion_cliente_sk
            ,MAX_BY(c.sk_cliente, fecha) AS ultima_transaccion_cliente_sk
            ,SUM(CASE WHEN tipoTran = 'Venta' THEN 1 ELSE 0 END) AS total_compras
            ,SUM(CASE WHEN tipoTran = 'Compra' THEN 1 ELSE 0 END) AS total_ventas
            ,FIRST(moneda) AS moneda
            ,COUNT(*) AS total_transacciones
            FROM iol_challenge.silver.deduped_transactions d
                LEFT JOIN iol_challenge.gold.dim_clientes c
                    ON c.id_cliente = d.id_cliente
                GROUP BY simbolo_titulo
    )

Observamos qué hay muchos casos qué las listas fijas no lograron clasificar

In [0]:
%sql
SELECT simbolo_tipo, count(*) FROM iol_challenge.gold.dim_cotizaciones GROUP BY simbolo_tipo;

Construimos una tabla factual con todas las transacciones

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.fact_transaction AS (
    SELECT 
        t.id_cliente
        ,t.moneda
        ,t.precio
        ,t.cantidad
        ,t.origen

        ,t.fecha
        ,t.simbolo_titulo
        ,c.simbolo_tipo
    
        ,di.High
        ,di.Low
        ,di.Open
        ,di.Close
        ,di.Volume
        ,(di.High + di.Low) / 2 as valor_mercado_promedio

    FROM iol_challenge.silver.deduped_transactions t
        LEFT JOIN iol_challenge.gold.dim_cotizaciones c
            ON c.simbolo_titulo = t.simbolo_titulo
        LEFT JOIN iol_challenge.silver.deduped_instruments di
            ON t.simbolo_titulo = di.simbolo AND di.date <= t.fecha
            --- Tomamos la ultima fecha disponible con información de cotización qué sea anterior a la fecha correspondiente a la agregación
            QUALIFY ROW_NUMBER() OVER (PARTITION BY t.fecha, t.id_cliente, t.simbolo_titulo, t.tipoTran ORDER BY di.date DESC) = 1 
)

Construimos una tabla factual con las transacciones por fecha, por cliente, instrumento e origen, enriqueciendo con las información de cotizaciones diarias. En primera instancia se agrupa el volumen transaccionado en el periodo de tiempo y cliente en cuestión pero se pueden agregar más campos.

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.fact_transaction_daily AS (
WITH grouped_data AS (
    SELECT
        t.fecha::date
        ,t.id_cliente
        ,t.simbolo_titulo
        ,t.tipoTran
        ,t.origen
        ,SUM(CASE WHEN t.moneda='ARS' THEN t.cantidad * t.precio ELSE 0 END) AS volumen_transaccionado_ars
        ,SUM(CASE WHEN t.moneda='USD' THEN t.cantidad * t.precio ELSE 0 END) AS volumen_transaccionado_usd
        ,COUNT(*) as total_transactions
            FROM iol_challenge.silver.deduped_transactions t
        GROUP BY t.fecha::date, t.id_cliente, t.simbolo_titulo, t.tipoTran, t.origen
    )
    SELECT
        -- Información sobre la cotización del instrumento
        g.fecha
        ,g.id_cliente
        ,g.simbolo_titulo
        ,g.tipoTran
        ,g.volumen_transaccionado_ars
        ,g.volumen_transaccionado_usd
        ,g.origen
        ,g.total_transactions

        ,di.High
        ,di.Low
        ,di.Open
        ,di.Close
        ,di.Volume
        ,(di.High + di.Low) / 2 as valor_mercado_promedio
        
            FROM grouped_data g
                LEFT JOIN iol_challenge.silver.deduped_instruments di
                    ON g.simbolo_titulo = di.simbolo AND di.date <= g.fecha
            --- Tomamos la ultima fecha disponible con información de cotización qué sea anterior a la fecha correspondiente a la agregación
            QUALIFY ROW_NUMBER() OVER (PARTITION BY g.fecha, g.id_cliente, g.simbolo_titulo, g.tipoTran ORDER BY di.date DESC) = 1
)

In [0]:
%sql
SELECT * FROM iol_challenge.gold.fact_transaction_daily limit 10;